In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

print("=== INICIANDO PREPROCESAMIENTO ===\n")

print("--- 1. Carga de Datos Inicial ---")
df = pd.read_csv('C:/Users/basti/proyecto_cd/data/raw/spotify-tracks-dataset.csv')
print(f"Dimensiones iniciales del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.\n")

=== INICIANDO PREPROCESAMIENTO ===

--- 1. Carga de Datos Inicial ---
Dimensiones iniciales del dataset: 114000 filas y 22 columnas.



In [2]:
print("--- 2. Limpieza de Columnas Innecesarias ---")
# Identificamos columnas basura generadas por exportaciones previas
cols_to_drop = ['Unnamed: 0', 'Unnamed: 0.1']
cols_exist = [col for col in cols_to_drop if col in df.columns]

if cols_exist:
    df = df.drop(columns=cols_exist)
    print(f"Se identificaron y eliminaron las columnas redundantes: {cols_exist}")
print(f"Columnas resultantes tras la limpieza: {df.shape[1]}\n")

--- 2. Limpieza de Columnas Innecesarias ---
Se identificaron y eliminaron las columnas redundantes: ['Unnamed: 0', 'Unnamed: 0.1']
Columnas resultantes tras la limpieza: 20



In [3]:
print("--- 3. Identificación de Valores Nulos ---")
nulos_por_columna = df.isnull().sum()
nulos_totales = nulos_por_columna[nulos_por_columna > 0]

if not nulos_totales.empty:
    print("Se encontraron valores nulos en las siguientes columnas:")
    print(nulos_totales.to_string())
    # Eliminamos las filas afectadas, ya que representan una fracción minúscula
    df = df.dropna()
    print(f"-> Acción: Se eliminaron las filas con valores nulos.")
    print(f"-> Nuevas dimensiones: {df.shape[0]} filas.\n")
else:
    print("No se encontraron valores nulos en el dataset.\n")

--- 3. Identificación de Valores Nulos ---
Se encontraron valores nulos en las siguientes columnas:
artists       1
album_name    1
track_name    1
-> Acción: Se eliminaron las filas con valores nulos.
-> Nuevas dimensiones: 113999 filas.



In [4]:
print("--- 4. Identificación de Canciones Duplicadas ---")
# Usamos 'track_name' y 'artists' porque Spotify a veces asigna distintos track_id a la misma canción si aparece en un Álbum y luego en un Single.
duplicados_exactos = df.duplicated(subset=['track_name', 'artists']).sum()
print(f"Se identificaron {duplicados_exactos} canciones duplicadas.")

# Nos quedamos con la primera aparición y descartamos el resto
df = df.drop_duplicates(subset=['track_name', 'artists'], keep='first').copy()
print(f"-> Acción: Se eliminaron los duplicados para evitar sesgos en la recomendación.")
print(f"-> Dimensiones tras la limpieza: {df.shape[0]} filas únicas (canciones reales).\n")

--- 4. Identificación de Canciones Duplicadas ---
Se identificaron 32656 canciones duplicadas.
-> Acción: Se eliminaron los duplicados para evitar sesgos en la recomendación.
-> Dimensiones tras la limpieza: 81343 filas únicas (canciones reales).



In [5]:
print("--- 5. Transformación y Escalado de Variables ---")
features_to_scale = ['danceability', 'energy', 'loudness', 'speechiness', 
                     'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

print("Variables continuas a escalar para el cálculo de distancia espacial (KNN):")
print(features_to_scale)

# Aplicamos MinMaxScaler para comprimir los valores al rango [0, 1]
scaler = MinMaxScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])
print("-> Acción: Escalado MinMaxScaler aplicado exitosamente.\n")

--- 5. Transformación y Escalado de Variables ---
Variables continuas a escalar para el cálculo de distancia espacial (KNN):
['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
-> Acción: Escalado MinMaxScaler aplicado exitosamente.



In [9]:
import os

print("--- 6. Exportación de Resultados ---")
# Definimos ruta específica
output_dir = r'C:/Users/basti/proyecto_cd/data/processed/'
output_filename = 'spotify_dataset_processed.csv'

full_path = os.path.join(output_dir, output_filename)

# Verificamos si la carpeta 'processed' existe; si no, la creamos automáticamente
os.makedirs(output_dir, exist_ok=True)

# Guardamos el archivo
df.to_csv(full_path, index=False)
print(f"¡ÉXITO! El dataset finalizado se ha guardado en:\n'{full_path}'")
print(f"Dimensiones finales listas para Machine Learning: {df.shape[0]} filas y {df.shape[1]} columnas.")
print("============================================")

df.head(100)

--- 6. Exportación de Resultados ---
¡ÉXITO! El dataset finalizado se ha guardado en:
'C:/Users/basti/proyecto_cd/data/processed/spotify_dataset_processed.csv'
Dimensiones finales listas para Machine Learning: 81343 filas y 20 columnas.


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.686294,0.4610,1,0.791392,0,0.148187,0.032329,0.000001,0.3580,0.718593,0.361245,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.426396,0.1660,1,0.597377,1,0.079067,0.927711,0.000006,0.1010,0.268342,0.318397,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.444670,0.3590,0,0.736123,1,0.057720,0.210843,0.000000,0.1170,0.120603,0.313643,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.270051,0.0596,0,0.573701,1,0.037617,0.908635,0.000071,0.1320,0.143719,0.746758,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.627411,0.4430,2,0.737103,1,0.054508,0.470884,0.000000,0.0829,0.167839,0.492863,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,7BRCa8MPiyuvr2VU3O9W0F,Joshua Hyslop,Where The Mountain Meets The Valley,Do Not Let Me Go,60,158960,False,0.415228,0.2340,3,0.662560,1,0.033472,0.339357,0.000050,0.0895,0.145729,0.574561,4,acoustic
123,5ZhaKUsY68U0lgREFWfoxg,Ross Copperman,Holding On And Letting Go - Single,Holding On And Letting Go,49,319000,False,0.509645,0.5300,2,0.739101,1,0.027565,0.808233,0.000738,0.0728,0.102513,0.320686,4,acoustic
124,4zcMBfbQZvBSHQBGDd6gsN,JJ Heller,You Already Know,You Already Know,57,214360,False,0.503553,0.1170,0,0.677857,1,0.031192,0.936747,0.000000,0.1260,0.319598,0.320776,4,acoustic
125,7bhHLZxkRekrNPPkEdDTbn,Ben Woodward,Believer (Remix),Believer (Remix),42,204480,False,0.593909,0.7240,1,0.795701,0,0.077202,0.316265,0.068100,0.0448,0.334673,0.770360,3,acoustic
